# Grad-CAM: Visualizing What CNNs See

**Gradient-weighted Class Activation Mapping (Grad-CAM)** is a powerful technique for understanding what convolutional neural networks "see" when making predictions.

## What We'll Learn

- How Grad-CAM uses gradients to weight activation maps
- The mathematical intuition behind gradient-weighted visualization
- Implementing Grad-CAM from scratch in PyTorch
- Comparing Grad-CAM with other visualization techniques
- Applications in model debugging, trust, and bias detection

## Why This Matters

Deep learning models are often criticized as "black boxes." Grad-CAM opens this box by:
- **Debugging**: Seeing if the model focuses on relevant features
- **Trust**: Verifying the model looks at appropriate image regions
- **Bias Detection**: Discovering spurious correlations in training data
- **Interpretability**: Explaining predictions to non-technical stakeholders

## Key Intuitions

1. **Activation Maps**: CNNs create spatial feature maps at each layer
2. **Gradients as Importance**: Higher gradients indicate more important features
3. **Weighted Combination**: Multiply activations by their importance weights
4. **Spatial Localization**: Resulting heatmap shows "where" the model looks

## 1. Setup

### Import Required Libraries

We'll use PyTorch for deep learning, our shared library for utilities, and matplotlib for visualization.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

from aiml_notebooks import (
    get_device,
    set_seed,
    CIFAR10_CLASSES,
    CIFAR10_MEAN,
    CIFAR10_STD,
)

%load_ext autoreload
%autoreload 2

### Configure Environment

Set random seed for reproducibility and configure device (GPU if available).

In [ ]:
set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 2. Understanding the Problem: Why Visualize CNNs?

### The Black Box Challenge

Neural networks can classify images with high accuracy, but **how** do they make decisions? Consider these critical questions:

1. **Medical Diagnosis**: Does the model detect cancer from the tumor or from metadata labels?
2. **Autonomous Vehicles**: Does the model recognize stop signs or just red octagonal shapes?
3. **Security Systems**: Does face recognition work on faces or backgrounds?

Without visualization techniques, we're flying blind. Let's see why we need more than just accuracy.

### Load a Pre-trained Model

We'll use a small CNN trained on CIFAR-10. This lets us focus on visualization rather than training.

In [ ]:
class SimpleCNN(nn.Module):
    """A simple CNN for CIFAR-10 classification."""
    
    def __init__(self, num_classes=10):
        super().__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(512)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
        
        # Fully connected layers
        self.fc1 = nn.Linear(512 * 2 * 2, 256)
        self.fc2 = nn.Linear(256, num_classes)
        
    def forward(self, x):
        # Layer 1: 32x32 -> 16x16
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        # Layer 2: 16x16 -> 8x8
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        # Layer 3: 8x8 -> 4x4
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        # Layer 4: 4x4 -> 2x2
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)
print(f"Model has {sum(p.numel() for p in model.parameters())} parameters")

### Load CIFAR-10 Dataset

We'll use CIFAR-10 for demonstration. It contains 32x32 color images across 10 classes.

In [ ]:
# Transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

# Load test set
test_dataset = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# Create a small subset for quick experimentation
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

print(f"Test set size: {len(test_dataset)}")
print(f"Classes: {CIFAR10_CLASSES}")

### Quick Training (Optional)

For demonstration purposes, we'll train the model briefly. In practice, you'd use a pre-trained model or train longer.

In [ ]:
# Load training data
train_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Quick training (3 epochs for demonstration)
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if i % 100 == 99:
            print(f"Epoch {epoch+1}/{num_epochs}, Batch {i+1}, "
                  f"Loss: {running_loss/100:.3f}, "
                  f"Acc: {100.*correct/total:.2f}%")
            running_loss = 0.0

print("\nTraining complete!")

### Test Model Accuracy

Let's verify the model works reasonably well before visualizing its decisions.

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

accuracy = 100. * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

## 3. Understanding Activation Maps

### What Are Activation Maps?

**Activation maps** (also called feature maps) are the outputs of convolutional layers. Each map shows where certain features are detected in the input image.

For example:
- Early layers detect edges, colors, textures
- Middle layers detect parts (wheels, eyes, wings)
- Late layers detect objects (cars, faces, birds)

The key insight: **spatial information is preserved** through convolutions, unlike fully connected layers.

### Extracting Activation Maps

Let's extract and visualize activation maps from the last convolutional layer.

In [ ]:
# Get a sample image
sample_image, sample_label = test_dataset[0]
sample_image = sample_image.unsqueeze(0).to(device)  # Add batch dimension

print(f"Image shape: {sample_image.shape}")
print(f"True label: {CIFAR10_CLASSES[sample_label]}")

### Hook for Capturing Activations

PyTorch **hooks** let us capture intermediate layer outputs during forward pass without modifying the model.

In [ ]:
# Storage for activations
activations = {}

def get_activation(name):
    """Returns a hook function that stores activations."""
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

# Register hook on the last conv layer (before pooling)
handle = model.conv4.register_forward_hook(get_activation('conv4'))

# Forward pass
model.eval()
with torch.no_grad():
    output = model(sample_image)
    
predicted_class = output.argmax(dim=1).item()
print(f"Predicted: {CIFAR10_CLASSES[predicted_class]}")
print(f"Activation map shape: {activations['conv4'].shape}")  # [1, 512, 4, 4]

### Visualize Activation Maps

Let's visualize a few of the 512 activation maps from the last convolutional layer.

In [ ]:
def denormalize(img):
    """Denormalize CIFAR-10 image for display."""
    img = img * torch.tensor(CIFAR10_STD).view(3, 1, 1) + torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
    return img.clamp(0, 1)

# Plot original image and some activation maps
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Original image
original = denormalize(sample_image[0].cpu())
axes[0, 0].imshow(original.permute(1, 2, 0))
axes[0, 0].set_title(f"Original\n{CIFAR10_CLASSES[sample_label]}")
axes[0, 0].axis('off')

# Show first 9 activation maps
act_maps = activations['conv4'][0].cpu().numpy()  # Shape: [512, 4, 4]
for i in range(9):
    row = (i + 1) // 5
    col = (i + 1) % 5
    axes[row, col].imshow(act_maps[i * 50], cmap='viridis')  # Sample every 50th map
    axes[row, col].set_title(f"Channel {i*50}")
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

### Key Observations

Notice how:
- Different channels activate in different regions
- Some channels are very selective (sparse activation)
- Others respond broadly (dense activation)

**Challenge**: Which channels are important for the final prediction? This is what Grad-CAM solves.

## 4. The Gradient Insight

### Why Gradients Matter

The **gradient** of the prediction score with respect to a feature map tells us:

$$\frac{\partial y^c}{\partial A^k}$$

Where:
- $y^c$ is the score for class $c$
- $A^k$ is the $k$-th activation map

**Intuition**: If a small change in $A^k$ causes a large change in $y^c$, then $A^k$ is important for predicting class $c$.

This is the foundation of Grad-CAM: **use gradients to weight the importance of activation maps**.

### Computing Gradients

Let's compute gradients of the predicted class score with respect to the last convolutional layer.

In [ ]:
# Storage for gradients
gradients = {}

def get_gradient(name):
    """Returns a hook function that stores gradients."""
    def hook(module, grad_input, grad_output):
        gradients[name] = grad_output[0].detach()
    return hook

# Register backward hook
handle_grad = model.conv4.register_full_backward_hook(get_gradient('conv4'))

# Forward pass (need to track gradients this time)
model.zero_grad()
output = model(sample_image)

# Backward pass for the predicted class
target_class = output.argmax(dim=1)
output[0, target_class].backward()

print(f"Gradients shape: {gradients['conv4'].shape}")  # Same as activations: [1, 512, 4, 4]

### Visualize Gradients

Let's see how gradients differ from activations.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))

# Original image
axes[0, 0].imshow(original.permute(1, 2, 0))
axes[0, 0].set_title("Original")
axes[0, 0].axis('off')

grad_maps = gradients['conv4'][0].cpu().numpy()

for i in range(4):
    # Activation
    axes[1, i+1].imshow(act_maps[i * 100], cmap='viridis')
    axes[1, i+1].set_title(f"Activation {i*100}")
    axes[1, i+1].axis('off')
    
    # Gradient
    axes[2, i+1].imshow(grad_maps[i * 100], cmap='coolwarm')
    axes[2, i+1].set_title(f"Gradient {i*100}")
    axes[2, i+1].axis('off')

# Hide unused subplots
for i in range(1, 5):
    axes[0, i].axis('off')
axes[1, 0].axis('off')
axes[2, 0].axis('off')

plt.tight_layout()
plt.show()

### Key Insight

Gradients show **importance** (high gradient = important for prediction), while activations show **presence** (high activation = feature detected).

**Grad-CAM combines both**: important features that are actually present in the image.

## 5. Grad-CAM Algorithm

### The Mathematical Formula

Grad-CAM computes a class-discriminative localization map $L_{Grad-CAM}^c$ for class $c$ as:

$$L_{Grad-CAM}^c = ReLU\left(\sum_k \alpha_k^c A^k\right)$$

Where the importance weights $\alpha_k^c$ are:

$$\alpha_k^c = \frac{1}{Z}\sum_i \sum_j \frac{\partial y^c}{\partial A_{ij}^k}$$

**Breaking it down**:
1. Compute gradients of class score $y^c$ w.r.t. feature maps $A^k$
2. Global average pool gradients to get weights $\alpha_k^c$ (one per channel)
3. Weight each feature map by its importance
4. Sum weighted feature maps
5. Apply ReLU to focus on positive contributions

The ReLU is crucial: we only care about features that **increase** the class score, not decrease it.

### Step 1: Global Average Pooling of Gradients

Average gradients across spatial dimensions to get per-channel importance weights.

In [ ]:
# Gradients shape: [1, 512, 4, 4]
# Average over spatial dimensions (4x4) -> [1, 512]
weights = gradients['conv4'].mean(dim=(2, 3), keepdim=True)

print(f"Weights shape: {weights.shape}")  # [1, 512, 1, 1]
print(f"First 5 weights: {weights[0, :5, 0, 0]}")

### Step 2: Weighted Combination of Activation Maps

Multiply each activation map by its importance weight and sum across channels.

In [ ]:
# Weighted sum: [1, 512, 4, 4] * [1, 512, 1, 1] -> [1, 512, 4, 4] -> sum -> [1, 4, 4]
weighted_activations = activations['conv4'] * weights
cam = weighted_activations.sum(dim=1, keepdim=True)

print(f"CAM shape: {cam.shape}")  # [1, 1, 4, 4]

### Step 3: Apply ReLU

Keep only positive contributions (features that increase the class score).

In [ ]:
cam = F.relu(cam)
print(f"CAM after ReLU - min: {cam.min():.3f}, max: {cam.max():.3f}")

### Step 4: Normalize and Upscale

Normalize to [0, 1] range and upscale to original image size for overlay.

In [ ]:
# Normalize to [0, 1]
cam = cam - cam.min()
cam = cam / (cam.max() + 1e-8)

# Upscale from 4x4 to 32x32 (original image size)
cam = F.interpolate(cam, size=(32, 32), mode='bilinear', align_corners=False)
cam = cam.squeeze().cpu().numpy()

print(f"Final CAM shape: {cam.shape}")  # [32, 32]
print(f"Final CAM range: [{cam.min():.3f}, {cam.max():.3f}]")

### Visualize Grad-CAM Heatmap

Let's see the heatmap overlaid on the original image.

In [ ]:
def apply_colormap_on_image(img, cam, alpha=0.5):
    """Overlay CAM heatmap on image."""
    # Convert to numpy
    img_np = img.permute(1, 2, 0).numpy()
    
    # Apply colormap to CAM
    heatmap = cm.jet(cam)[:, :, :3]  # RGB, no alpha
    
    # Overlay
    overlaid = heatmap * alpha + img_np * (1 - alpha)
    overlaid = np.clip(overlaid, 0, 1)
    
    return overlaid, heatmap

# Create overlay
overlaid, heatmap = apply_colormap_on_image(original, cam)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original.permute(1, 2, 0))
axes[0].set_title(f"Original\n{CIFAR10_CLASSES[sample_label]}")
axes[0].axis('off')

axes[1].imshow(heatmap)
axes[1].set_title(f"Grad-CAM Heatmap\nPredicted: {CIFAR10_CLASSES[predicted_class]}")
axes[1].axis('off')

axes[2].imshow(overlaid)
axes[2].set_title("Overlay")
axes[2].axis('off')

plt.tight_layout()
plt.show()

### Understanding the Visualization

The heatmap shows:
- **Red regions**: High importance for the prediction
- **Blue regions**: Low importance

This reveals **where** the model is looking to make its decision.

## 6. Implementing a Reusable Grad-CAM Class

### Building a Clean Grad-CAM Implementation

Let's create a reusable class that encapsulates the Grad-CAM algorithm.

In [ ]:
class GradCAM:
    """Grad-CAM implementation for PyTorch models."""
    
    def __init__(self, model, target_layer):
        """
        Args:
            model: PyTorch model
            target_layer: Layer to compute CAM for (usually last conv layer)
        """
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        
        # Register hooks
        self.forward_handle = target_layer.register_forward_hook(self._save_activation)
        self.backward_handle = target_layer.register_full_backward_hook(self._save_gradient)
    
    def _save_activation(self, module, input, output):
        """Hook to save activations during forward pass."""
        self.activations = output.detach()
    
    def _save_gradient(self, module, grad_input, grad_output):
        """Hook to save gradients during backward pass."""
        self.gradients = grad_output[0].detach()
    
    def __call__(self, image, target_class=None):
        """
        Generate Grad-CAM for an image.
        
        Args:
            image: Input image tensor [1, C, H, W]
            target_class: Class to visualize (if None, uses predicted class)
            
        Returns:
            cam: Grad-CAM heatmap [H, W]
            predicted_class: Model's prediction
        """
        # Forward pass
        self.model.eval()
        self.model.zero_grad()
        output = self.model(image)
        
        # Get target class
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        
        # Backward pass
        output[0, target_class].backward()
        
        # Compute Grad-CAM
        # 1. Global average pooling of gradients
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        
        # 2. Weighted combination
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        
        # 3. ReLU
        cam = F.relu(cam)
        
        # 4. Normalize
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        
        # 5. Upscale to input size
        H, W = image.shape[2:]
        cam = F.interpolate(cam, size=(H, W), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        
        return cam, target_class
    
    def remove_hooks(self):
        """Remove registered hooks."""
        self.forward_handle.remove()
        self.backward_handle.remove()

print("GradCAM class defined successfully!")

### Using the Grad-CAM Class

Let's test our implementation on multiple images.

In [ ]:
# Create Grad-CAM instance
gradcam = GradCAM(model, model.conv4)

# Test on multiple images
fig, axes = plt.subplots(3, 6, figsize=(18, 9))

for i in range(6):
    # Get image
    img, label = test_dataset[i * 100]
    img_batch = img.unsqueeze(0).to(device)
    
    # Generate Grad-CAM
    cam, pred_class = gradcam(img_batch)
    
    # Denormalize for display
    img_display = denormalize(img)
    overlaid, heatmap = apply_colormap_on_image(img_display, cam)
    
    # Plot
    axes[0, i].imshow(img_display.permute(1, 2, 0))
    axes[0, i].set_title(f"{CIFAR10_CLASSES[label]}", fontsize=9)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(heatmap)
    axes[1, i].set_title(f"Pred: {CIFAR10_CLASSES[pred_class]}", fontsize=9)
    axes[1, i].axis('off')
    
    axes[2, i].imshow(overlaid)
    axes[2, i].set_title("Overlay", fontsize=9)
    axes[2, i].axis('off')

plt.tight_layout()
plt.show()

gradcam.remove_hooks()

### Analyzing the Results

Look at where the model focuses:
- Does it focus on the object or the background?
- For misclassifications, what did the model look at?
- Are there any surprising patterns?

This is where Grad-CAM becomes a debugging tool.

## 7. Comparing Visualization Methods

### Alternative Visualization Techniques

Grad-CAM is not the only visualization method. Let's compare it with:

1. **Vanilla Gradients**: $\frac{\partial y^c}{\partial X}$ (gradient of class score w.r.t. input)
2. **Guided Backpropagation**: Modified backprop that only passes positive gradients
3. **Integrated Gradients**: Accumulates gradients along a path from baseline to input

Each method has different strengths and weaknesses.

### Method 1: Vanilla Gradients

The simplest approach: compute gradient of the class score with respect to the input image.

In [ ]:
def vanilla_gradients(model, image, target_class=None):
    """Compute vanilla gradients for visualization."""
    model.eval()
    image.requires_grad = True
    model.zero_grad()
    
    # Forward pass
    output = model(image)
    
    if target_class is None:
        target_class = output.argmax(dim=1).item()
    
    # Backward pass
    output[0, target_class].backward()
    
    # Get gradients
    gradients = image.grad.detach().cpu()
    
    # Take absolute value and average across channels
    gradients = gradients.abs().mean(dim=1).squeeze().numpy()
    
    # Normalize
    gradients = gradients - gradients.min()
    gradients = gradients / (gradients.max() + 1e-8)
    
    image.requires_grad = False
    
    return gradients, target_class

# Test
sample_image.requires_grad = True
vanilla_grad, _ = vanilla_gradients(model, sample_image)
print(f"Vanilla gradients shape: {vanilla_grad.shape}")

### Method 2: Guided Backpropagation

Guided backprop modifies ReLU to only backpropagate positive gradients, creating sharper visualizations.

In [ ]:
class GuidedBackprop:
    """Guided backpropagation implementation."""
    
    def __init__(self, model):
        self.model = model
        self.handles = []
        self._register_hooks()
    
    def _register_hooks(self):
        """Register hooks on all ReLU layers."""
        def backward_hook(module, grad_in, grad_out):
            # Only pass positive gradients
            return (F.relu(grad_in[0]),)
        
        for module in self.model.modules():
            if isinstance(module, nn.ReLU):
                handle = module.register_full_backward_hook(backward_hook)
                self.handles.append(handle)
    
    def __call__(self, image, target_class=None):
        """Generate guided backprop visualization."""
        self.model.eval()
        image.requires_grad = True
        self.model.zero_grad()
        
        # Forward pass
        output = self.model(image)
        
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        
        # Backward pass
        output[0, target_class].backward()
        
        # Get gradients
        gradients = image.grad.detach().cpu()
        gradients = gradients.abs().mean(dim=1).squeeze().numpy()
        
        # Normalize
        gradients = gradients - gradients.min()
        gradients = gradients / (gradients.max() + 1e-8)
        
        image.requires_grad = False
        
        return gradients, target_class
    
    def remove_hooks(self):
        """Remove registered hooks."""
        for handle in self.handles:
            handle.remove()

# Test
guided_bp = GuidedBackprop(model)
guided_grad, _ = guided_bp(sample_image)
print(f"Guided backprop shape: {guided_grad.shape}")
guided_bp.remove_hooks()

### Compare All Methods Side-by-Side

Let's visualize vanilla gradients, guided backprop, and Grad-CAM together.

In [ ]:
# Generate visualizations
gradcam = GradCAM(model, model.conv4)
guided_bp = GuidedBackprop(model)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for i in range(4):
    # Get image
    img, label = test_dataset[i * 200]
    img_batch = img.unsqueeze(0).to(device)
    img_display = denormalize(img)
    
    # 1. Vanilla gradients
    img_batch.requires_grad = True
    vanilla_grad, pred = vanilla_gradients(model, img_batch)
    img_batch.requires_grad = False
    
    # 2. Guided backprop
    guided_grad, _ = guided_bp(img_batch)
    
    # 3. Grad-CAM
    cam, _ = gradcam(img_batch)
    overlaid, _ = apply_colormap_on_image(img_display, cam)
    
    # Plot
    axes[i, 0].imshow(img_display.permute(1, 2, 0))
    axes[i, 0].set_title(f"Original: {CIFAR10_CLASSES[label]}\nPred: {CIFAR10_CLASSES[pred]}")
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(vanilla_grad, cmap='hot')
    axes[i, 1].set_title("Vanilla Gradients")
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(guided_grad, cmap='hot')
    axes[i, 2].set_title("Guided Backprop")
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow(overlaid)
    axes[i, 3].set_title("Grad-CAM")
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

gradcam.remove_hooks()
guided_bp.remove_hooks()

### Comparing the Methods

**Vanilla Gradients**:
- Pros: Simple, no modification needed
- Cons: Noisy, sensitive to small perturbations

**Guided Backpropagation**:
- Pros: Sharper, cleaner visualizations
- Cons: Can be too sharp, loses spatial localization

**Grad-CAM**:
- Pros: Class-discriminative, good localization, interpretable
- Cons: Lower resolution (limited by feature map size)

**Best practice**: Use Grad-CAM for "where" and guided backprop for "what".

## 8. Guided Grad-CAM: Best of Both Worlds

### Combining Grad-CAM and Guided Backprop

**Guided Grad-CAM** multiplies the Grad-CAM heatmap with guided backpropagation to get:
- Spatial localization from Grad-CAM
- Fine-grained detail from guided backprop

This gives us the best of both worlds.

In [ ]:
def guided_gradcam(gradcam, guided_bp, image):
    """Combine Grad-CAM and Guided Backpropagation."""
    # Get Grad-CAM
    cam, pred = gradcam(image)
    
    # Get Guided Backprop
    guided_grad, _ = guided_bp(image)
    
    # Element-wise multiplication
    guided_cam = cam * guided_grad
    
    # Normalize
    guided_cam = guided_cam - guided_cam.min()
    guided_cam = guided_cam / (guided_cam.max() + 1e-8)
    
    return guided_cam, pred

# Test
gradcam = GradCAM(model, model.conv4)
guided_bp = GuidedBackprop(model)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i in range(5):
    img, label = test_dataset[i * 150]
    img_batch = img.unsqueeze(0).to(device)
    img_display = denormalize(img)
    
    # Grad-CAM
    cam, pred = gradcam(img_batch)
    overlaid, _ = apply_colormap_on_image(img_display, cam)
    
    # Guided Grad-CAM
    guided_cam, _ = guided_gradcam(gradcam, guided_bp, img_batch)
    
    # Plot
    axes[0, i].imshow(overlaid)
    axes[0, i].set_title(f"Grad-CAM\n{CIFAR10_CLASSES[label]}")
    axes[0, i].axis('off')
    
    axes[1, i].imshow(guided_cam, cmap='hot')
    axes[1, i].set_title(f"Guided Grad-CAM\nPred: {CIFAR10_CLASSES[pred]}")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

gradcam.remove_hooks()
guided_bp.remove_hooks()

### When to Use Each Method

- **Grad-CAM**: Quick debugging, understanding spatial focus
- **Guided Backprop**: Detailed feature visualization
- **Guided Grad-CAM**: Publication-quality visualizations, detailed analysis

For most practical debugging, **Grad-CAM alone is sufficient**.

## 9. Applications: Debugging and Trust

### Application 1: Finding Misclassifications

Let's use Grad-CAM to understand **why** the model makes mistakes.

In [ ]:
# Find misclassified examples
model.eval()
misclassified = []

with torch.no_grad():
    for i in range(len(test_dataset)):
        img, label = test_dataset[i]
        img_batch = img.unsqueeze(0).to(device)
        output = model(img_batch)
        pred = output.argmax(dim=1).item()
        
        if pred != label:
            misclassified.append((i, label, pred))
        
        if len(misclassified) >= 6:
            break

print(f"Found {len(misclassified)} misclassifications")

### Visualize Misclassifications with Grad-CAM

Let's see where the model was looking when it made these mistakes.

In [ ]:
gradcam = GradCAM(model, model.conv4)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))

for i, (idx, true_label, pred_label) in enumerate(misclassified):
    img, _ = test_dataset[idx]
    img_batch = img.unsqueeze(0).to(device)
    img_display = denormalize(img)
    
    # Generate Grad-CAM
    cam, _ = gradcam(img_batch)
    overlaid, _ = apply_colormap_on_image(img_display, cam)
    
    # Plot
    axes[0, i].imshow(img_display.permute(1, 2, 0))
    axes[0, i].set_title(f"True: {CIFAR10_CLASSES[true_label]}", fontsize=9)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(overlaid)
    axes[1, i].set_title(f"Pred: {CIFAR10_CLASSES[pred_label]}", fontsize=9, color='red')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

gradcam.remove_hooks()

### Insights from Misclassifications

Look at the heatmaps:
- Is the model focusing on the wrong object?
- Is the image ambiguous (could be multiple classes)?
- Is the model using spurious correlations (e.g., background)?

This helps us:
1. **Improve data**: Remove ambiguous samples
2. **Fix biases**: Identify and remove spurious correlations
3. **Improve architecture**: Target specific failure modes

### Application 2: Class Discrimination

Grad-CAM can visualize **any** class, not just the predicted one. This helps understand what the model associates with each class.

In [ ]:
# Pick an image and visualize different class activations
img, label = test_dataset[42]
img_batch = img.unsqueeze(0).to(device)
img_display = denormalize(img)

gradcam = GradCAM(model, model.conv4)

# Get top-5 predictions
model.eval()
with torch.no_grad():
    output = model(img_batch)
    probs = F.softmax(output, dim=1)
    top5_probs, top5_classes = probs.topk(5, dim=1)
    top5_probs = top5_probs[0].cpu().numpy()
    top5_classes = top5_classes[0].cpu().numpy()

# Visualize Grad-CAM for each top-5 class
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Original image
axes[0, 0].imshow(img_display.permute(1, 2, 0))
axes[0, 0].set_title(f"Original\nTrue: {CIFAR10_CLASSES[label]}")
axes[0, 0].axis('off')

# Grad-CAM for top-5 classes
for i in range(5):
    class_idx = top5_classes[i]
    prob = top5_probs[i]
    
    # Generate Grad-CAM for this class
    cam, _ = gradcam(img_batch, target_class=class_idx)
    overlaid, _ = apply_colormap_on_image(img_display, cam)
    
    row = (i + 1) // 3
    col = (i + 1) % 3
    axes[row, col].imshow(overlaid)
    axes[row, col].set_title(f"{CIFAR10_CLASSES[class_idx]}\n{prob*100:.1f}%")
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

gradcam.remove_hooks()

### Understanding Class Discrimination

Notice how:
- Different classes focus on different image regions
- The predicted class has strongest activation on the actual object
- Confused classes might focus on similar or overlapping regions

This reveals **what features the model uses to distinguish classes**.

## 10. Detecting Bias with Grad-CAM

### The Importance of Bias Detection

Models can learn **spurious correlations** from training data:
- Husky vs. Wolf: Model uses snow background instead of animal features
- Hospital X-rays: Model uses hospital logo instead of pathology
- Skin lesions: Model uses rulers/markers instead of lesion characteristics

Grad-CAM helps **detect these biases** before deployment.

### Systematic Bias Analysis

Let's analyze where the model focuses across an entire class.

In [ ]:
def analyze_class_focus(model, dataset, class_idx, num_samples=20):
    """Aggregate Grad-CAM heatmaps for a specific class."""
    gradcam = GradCAM(model, model.conv4)
    
    # Find samples of this class
    class_indices = [i for i, (_, label) in enumerate(dataset) if label == class_idx]
    sample_indices = class_indices[:num_samples]
    
    # Accumulate heatmaps
    accumulated_cam = np.zeros((32, 32))
    
    for idx in sample_indices:
        img, _ = dataset[idx]
        img_batch = img.unsqueeze(0).to(device)
        
        cam, _ = gradcam(img_batch, target_class=class_idx)
        accumulated_cam += cam
    
    # Average
    accumulated_cam /= num_samples
    
    gradcam.remove_hooks()
    
    return accumulated_cam

# Analyze two different classes
airplane_focus = analyze_class_focus(model, test_dataset, class_idx=0, num_samples=20)
automobile_focus = analyze_class_focus(model, test_dataset, class_idx=1, num_samples=20)

print("Class focus analysis complete!")

### Visualize Average Class Focus

This shows the "typical" focus pattern for each class.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(airplane_focus, cmap='jet')
axes[0].set_title(f"Average Focus: {CIFAR10_CLASSES[0]}")
axes[0].axis('off')

axes[1].imshow(automobile_focus, cmap='jet')
axes[1].set_title(f"Average Focus: {CIFAR10_CLASSES[1]}")
axes[1].axis('off')

plt.tight_layout()
plt.show()

### Interpreting Average Focus

The average heatmap reveals:
- **Center focus**: Model looks at center regardless of object location (potential bias)
- **Edge focus**: Model uses image borders (data artifact)
- **Distributed focus**: Model uses diverse features (good generalization)

In production systems, this analysis should be done **before deployment** to catch dataset biases.

## 11. Limitations and Best Practices

### Limitations of Grad-CAM

While powerful, Grad-CAM has limitations:

1. **Resolution**: Limited by feature map size (4x4 in our case)
2. **Layer Choice**: Only visualizes one layer at a time
3. **Multiple Objects**: Can struggle with multiple objects of same class
4. **Architecture Specific**: Works best with CNNs (not Transformers)
5. **Interpretation**: High activation doesn't guarantee the model is "right"

**Don't** use Grad-CAM as the only validation method. Combine it with quantitative metrics.

### Best Practices

1. **Choose the Right Layer**: Use the last convolutional layer for best spatial resolution
2. **Validate Systematically**: Don't just cherry-pick good examples
3. **Compare Classes**: Look at Grad-CAM for multiple classes, not just the predicted one
4. **Test Edge Cases**: Focus on misclassifications and low-confidence predictions
5. **Document Biases**: Record and report any biases discovered
6. **Combine Methods**: Use multiple visualization techniques (Grad-CAM + saliency, etc.)
7. **Domain Experts**: Have domain experts review visualizations

**Remember**: Visualization is a tool for understanding, not a substitute for rigorous testing.

### When to Use Grad-CAM

**Use Grad-CAM when**:
- Debugging model predictions
- Explaining decisions to stakeholders
- Detecting dataset biases
- Comparing different models
- Building trust in deployed systems

**Don't use Grad-CAM as**:
- The sole evaluation metric
- A replacement for proper validation
- A guarantee of model correctness

## 12. Extensions and Advanced Topics

### Grad-CAM Variants

Several extensions improve upon vanilla Grad-CAM:

1. **Grad-CAM++**: Better localization for multiple objects
2. **Score-CAM**: Removes gradient dependency (more stable)
3. **Ablation-CAM**: Uses ablation studies instead of gradients
4. **XGrad-CAM**: Weighted gradients for better visualization
5. **Layer-CAM**: Combines multiple layers

Each has tradeoffs in speed, accuracy, and interpretability.

### Adapting Grad-CAM to Other Architectures

Grad-CAM can be adapted to:
- **Vision Transformers**: Visualize attention maps + Grad-CAM
- **Object Detection**: Generate Grad-CAM for bounding box predictions
- **Semantic Segmentation**: Pixel-wise Grad-CAM
- **Video Models**: Temporal Grad-CAM across frames

The core principle (gradient-weighted activation maps) remains the same.

### Quantifying Interpretability

Recent work tries to **quantify** visualization quality:
- **Deletion/Insertion**: Remove/add pixels by importance, measure accuracy drop
- **Pointing Game**: Measure if heatmap peaks align with object locations
- **Localization Accuracy**: Compare with ground-truth bounding boxes

This moves interpretability from subjective to objective evaluation.

## 13. Key Takeaways

### Core Concepts

1. **Grad-CAM** uses gradients to weight activation maps, revealing what CNNs "see"
2. The **algorithm**: forward pass → backward pass → global average pooling → weighted sum → ReLU → upscale
3. **Gradients measure importance**, activations measure presence
4. Grad-CAM is **class-discriminative**: can visualize any class, not just predictions

### Practical Applications

5. **Debugging**: Identify what the model focuses on
6. **Trust**: Verify model uses correct features
7. **Bias Detection**: Discover spurious correlations
8. **Comparison**: Different methods (vanilla, guided, Grad-CAM) complement each other

### Important Limitations

9. **Resolution** is limited by feature map size
10. Visualization is a **tool**, not a substitute for rigorous evaluation

### Remember

> "Interpretability is not about making models explainable. It's about making them trustworthy through understanding."

Grad-CAM is one of the most practical tools for model interpretability, but it should be part of a broader validation strategy.

## Further Exploration

Try these experiments:
1. Apply Grad-CAM to different layers and compare results
2. Implement Grad-CAM++ for better multi-object localization
3. Use Grad-CAM on a pre-trained ImageNet model
4. Quantify visualization quality using deletion/insertion metrics
5. Explore Grad-CAM for object detection or segmentation tasks

The techniques learned here generalize to any CNN-based vision model!